In [ ]:
# Cell 0: installs + ALL LoRA hyperparameters (the only place they are defined).
# Run on a cloud GPU (Kaggle/Colab T4, RunPod fallback). Upload train.jsonl and dev.jsonl from H7 `export` to DATA_DIR.
# !pip install -q unsloth trl datasets

DATA_DIR = "."
BASE_MODEL = "unsloth/gemma-3-1b-it"   # set to the bake-off winner's HF repo (not decided in this notebook)
OUT_DIR = "orion-lm-merged"            # 16-bit merged model; convert locally with train.convert.to_mlx

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
LEARNING_RATE = 2e-4
EPOCHS = 2
BATCH_SIZE = 4
GRAD_ACCUM = 4
MAX_SEQ_LEN = 8192
SEED = 3407
# These are starting values, not tuned or measured results.


In [ ]:
# Cell 1: data, prompt rendering, target-only loss mask, model loader.
import json, math, os, time

# VENDORED from orion/lm/model.py (the runtime applies it before every chat-template call). Copied verbatim
# rather than pip-installing the repo: the cloud box then needs no repo access or credentials, and the
# repo test test_vendored_merge_equals_repo_function fails if this copy ever drifts from the source.
def merge_consecutive_roles(messages: list[dict]) -> list[dict]:
    """Join runs of consecutive same-role messages into one message (contents joined by a blank line).

    Applied before every chat-template call, in training and at runtime alike. Returns new dicts; the
    caller's (append-only) history is never mutated. Idempotent.
    """
    out: list[dict] = []
    for m in messages:
        if out and out[-1]["role"] == m["role"]:
            out[-1] = {**out[-1], "content": out[-1]["content"] + "\n\n" + m["content"]}
        else:
            out.append(dict(m))
    return out


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(ln) for ln in f if ln.strip()]

train_rows = read_jsonl(os.path.join(DATA_DIR, "train.jsonl"))
dev_rows = read_jsonl(os.path.join(DATA_DIR, "dev.jsonl"))
print(f"train rows: {len(train_rows)}  dev rows: {len(dev_rows)}")

def load_model():
    from unsloth import FastLanguageModel
    model, tok = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True)
    model = FastLanguageModel.get_peft_model(
        model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=TARGET_MODULES, random_state=SEED)
    return model, tok

def encode(row, tok):
    """prompt = runtime prompt (merge_consecutive_roles, then chat template + generation prompt);
    completion = target + end-of-turn. Loss only on the completion: prompt labels are -100."""
    prompt = tok.apply_chat_template(merge_consecutive_roles(row["messages"]), tokenize=False,
                                     add_generation_prompt=True)
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    t_ids = tok(row["target"] + "<end_of_turn>", add_special_tokens=False)["input_ids"]
    ids = (p_ids + t_ids)[:MAX_SEQ_LEN]
    labels = ([-100] * len(p_ids) + t_ids)[:MAX_SEQ_LEN]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

def make_trainer(model, tok, rows, **extra):
    from datasets import Dataset
    from transformers import DataCollatorForSeq2Seq
    from trl import SFTConfig, SFTTrainer
    ds = Dataset.from_list([encode(r, tok) for r in rows])
    cfg = SFTConfig(output_dir="runs", per_device_train_batch_size=BATCH_SIZE,
                    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LEARNING_RATE,
                    num_train_epochs=EPOCHS, seed=SEED, logging_steps=10, save_strategy="no",
                    report_to="none", remove_unused_columns=False,
                    dataset_kwargs={"skip_prepare_dataset": True}, **extra)
    return SFTTrainer(model=model, args=cfg, train_dataset=ds,
                      data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))


In [ ]:
# Cell 2: throughput. Trains briefly on THROUGHPUT_N examples, then prints a PROJECTION for the full run.
# The projection is arithmetic on this one measurement only (no assumed speed, no other data).
THROUGHPUT_N = 200
model, tok = load_model()
sample = train_rows[:THROUGHPUT_N]
trainer = make_trainer(model, tok, sample, max_steps=-1)
trainer.args.num_train_epochs = 1
t0 = time.perf_counter()
trainer.train()
secs = time.perf_counter() - t0
rate = len(sample) / secs
print(f"measured: {len(sample)} examples in {secs:.1f} s = {rate:.3f} examples/s (1 epoch, includes warmup)")
full_examples = len(train_rows) * EPOCHS
print(f"PROJECTION (from the measurement above, not a measurement): {full_examples} example-passes "
      f"({len(train_rows)} rows x {EPOCHS} epochs) -> {full_examples / rate / 3600:.2f} h at {rate:.3f} examples/s")
del trainer, model  # the timing run trained the adapter; cell 3 starts from a fresh one


In [ ]:
# Cell 3: full fine-tune on train.jsonl (loss on target tokens only).
model, tok = load_model()
trainer = make_trainer(model, tok, train_rows)
trainer.train()


In [ ]:
# Cell 4: merge the LoRA into 16-bit weights and save. Download OUT_DIR, then locally:
#   from train.convert import to_mlx; to_mlx("orion-lm-merged", "orion-lm")        # fine-tune
#   from train.convert import to_mlx; to_mlx("<base HF dir>", "orion-lm-base")     # base, same call
model.save_pretrained_merged(OUT_DIR, tok, save_method="merged_16bit")
print("saved", OUT_DIR)
